In [2]:
import argparse
import pandas as pd
import pyhocon
from dataset import CrossEncoderDatasetFull

In [2]:
#Read NewsWCL50
config = pyhocon.ConfigFactory.parse_file('configs/config_pairwise.json')
temp_config = config.copy()
temp_config.put('data_folder',"../../CDCR/data/ECBplus-prep/output_data/")

#Train
#entities
temp_mention_type = "entity_mentions"
temp_config.put('mention_type',temp_mention_type)
newswcl50_entities_train = CrossEncoderDatasetFull(temp_config,split_name='train',negative_sampling=True)
#events
temp_mention_type = "event_mentions"
temp_config.put('mention_type',temp_mention_type)
newswcl50_events_train = CrossEncoderDatasetFull(temp_config,split_name='train',negative_sampling=True)

#Train
#entities
temp_mention_type = "entity_mentions"
temp_config.put('mention_type',temp_mention_type)
newswcl50_entities_val = CrossEncoderDatasetFull(temp_config,split_name='val',negative_sampling=True)
#events
temp_mention_type = "event_mentions"
temp_config.put('mention_type',temp_mention_type)
newswcl50_events_val = CrossEncoderDatasetFull(temp_config,split_name='val',negative_sampling=True)

#Test
#entities
temp_mention_type = "entity_mentions"
temp_config.put('mention_type',temp_mention_type)
newswcl50_entities_test = CrossEncoderDatasetFull(temp_config,split_name='test',negative_sampling=True)
#events
temp_mention_type = "event_mentions"
temp_config.put('mention_type',temp_mention_type)
newswcl50_events_test = CrossEncoderDatasetFull(temp_config,split_name='test',negative_sampling=True)

list_newswcl50_datasets = [newswcl50_entities_train,newswcl50_events_train,
                            newswcl50_entities_val, newswcl50_events_val,
                            newswcl50_entities_test, newswcl50_events_test]
list_entity_types = ['entities','events','entities','events','entities','events']

list_dfs_mentions = []

for dataset, ent_type in zip (list_newswcl50_datasets,list_entity_types):
    #mentiones entities
    mention_ids = []
    subtopic_ids = []
    coref_chains = []
    for mention in dataset.mentions:
        mention_ids.append(mention['mention_id'])
        subtopic_ids.append(mention['subtopic_id'])
        coref_chains.append(mention['coref_chain'])

    df_temp = pd.DataFrame({'mention_id':mention_ids,'subtopic':subtopic_ids,'coref_chain':coref_chains,'mention_type':[ent_type]*len(mention_ids)})
    list_dfs_mentions.append(df_temp.copy())

df_all_mentions = pd.concat(list_dfs_mentions)


c:\Users\sago\Documents\GitHub\CDCR_lexical_diversity\MTL\cross_encoder_adapted\dataset.py:139: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels,dtype=torch.float)


In [3]:
from enum import Enum
from pyhocon import ConfigFactory
import pandas as pd

# Define constants
ENTITY_MENTIONS = "entity_mentions"
EVENT_MENTIONS = "event_mentions"
SPLIT_NAMES = ['train', 'val', 'test']
DATA_FOLDER = "../../CDCR/data/ECBplus-prep/output_data/"

class EntityType(Enum):
    ENTITIES = "entities"
    EVENTS = "events"

# Read Config
config = ConfigFactory.parse_file('configs/config_pairwise.json')
config.put('data_folder', DATA_FOLDER)

def create_datasets(config, split_name):
    datasets = {}
    for mention_type, ent_type in ((ENTITY_MENTIONS, EntityType.ENTITIES), (EVENT_MENTIONS, EntityType.EVENTS)):
        config.put('mention_type', mention_type)
        datasets[ent_type] = CrossEncoderDatasetFull(config, split_name=split_name, negative_sampling=True)
    return datasets

datasets = {}
for split_name in SPLIT_NAMES:
    datasets[split_name] = create_datasets(config.copy(), split_name)

list_dfs_mentions = []

for split_name in SPLIT_NAMES:
    for ent_type, dataset in datasets[split_name].items():
        mention_ids = [mention['mention_id'] for mention in dataset.mentions]
        subtopic_ids = [mention['subtopic_id'] for mention in dataset.mentions]
        coref_chains = [mention['coref_chain'] for mention in dataset.mentions]
        
        df_temp = pd.DataFrame({
            'mention_id': mention_ids,
            'subtopic': subtopic_ids,
            'coref_chain': coref_chains,
            'mention_type': [ent_type.value] * len(mention_ids)
        })
        list_dfs_mentions.append(df_temp)

df_all_mentions = pd.concat(list_dfs_mentions)


c:\Users\sago\Documents\GitHub\CDCR_lexical_diversity\MTL\cross_encoder_adapted\dataset.py:139: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels,dtype=torch.float)


In [9]:
predicted_clusters =pd.read_excel("final_clusters_ecb.xlsx")

In [10]:
merged_values = predicted_clusters.merge(df_all_mentions,on='mention_id',how='inner')

In [7]:
# This is the critical part to ensure that the clusters are not mixed between combinations entity type and split
set_coref_chains = merged_values.coref_chain.unique()
enum_dict_comp = {item: index for index, item in enumerate(set_coref_chains)}
merged_values["mapped_coref_chain"] = merged_values["coref_chain"].map(enum_dict_comp)
# Adjust coref_chain and predicted clusters

exp1_seq_max,exp1_sim_max,exp2_seq_max,exp2_sim_max =0,0,0,0
exp3_seq_max,exp3_sim_max,exp4_seq_max,exp4_sim_max=0,0,0,0

unique_subtopics_keys = merged_values[['subtopic','mention_type']].drop_duplicates()
for index, row in unique_subtopics_keys.iterrows():
    df_temp = merged_values[(merged_values['subtopic']==row['subtopic'])&(merged_values['mention_type']==row['mention_type'])]
    
        # Use .loc for assignments to avoid the warning
    df_temp.loc[:, "cluster_exp1_seq_sampling"] += exp1_seq_max
    df_temp.loc[:, "cluster_exp1_sim_sampling"] += exp1_sim_max
    df_temp.loc[:, "cluster_exp2_seq_sampling"] += exp2_seq_max
    df_temp.loc[:, "cluster_exp2_sim_sampling"] += exp2_sim_max
    df_temp.loc[:, "cluster_exp3_seq_sampling"] += exp3_seq_max
    df_temp.loc[:, "cluster_exp3_sim_sampling"] += exp3_sim_max
    df_temp.loc[:, "cluster_exp4_seq_sampling"] += exp4_seq_max
    df_temp.loc[:, "cluster_exp4_sim_sampling"] += exp4_sim_max
    
    # Use .loc for assignments to avoid the warning
    merged_values.loc[df_temp.index, "cluster_exp1_seq_sampling"] += exp1_seq_max
    merged_values.loc[df_temp.index, "cluster_exp1_sim_sampling"] += exp1_sim_max
    merged_values.loc[df_temp.index, "cluster_exp2_seq_sampling"] += exp2_seq_max
    merged_values.loc[df_temp.index, "cluster_exp2_sim_sampling"] += exp2_sim_max
    merged_values.loc[df_temp.index, "cluster_exp3_seq_sampling"] += exp3_seq_max
    merged_values.loc[df_temp.index, "cluster_exp3_sim_sampling"] += exp3_sim_max
    merged_values.loc[df_temp.index, "cluster_exp4_seq_sampling"] += exp4_seq_max
    merged_values.loc[df_temp.index, "cluster_exp4_sim_sampling"] += exp4_sim_max

    exp1_seq_max = max(df_temp["cluster_exp1_seq_sampling"])
    exp1_sim_max = max(df_temp["cluster_exp1_sim_sampling"])
    exp2_seq_max = max(df_temp["cluster_exp2_seq_sampling"])
    exp2_sim_max = max(df_temp["cluster_exp2_sim_sampling"])
    exp3_seq_max = max(df_temp["cluster_exp3_seq_sampling"])
    exp3_sim_max = max(df_temp["cluster_exp3_sim_sampling"])
    exp4_seq_max = max(df_temp["cluster_exp4_seq_sampling"])
    exp4_sim_max = max(df_temp["cluster_exp4_sim_sampling"])

In [13]:
unique_subtopics_keys

,subtopic,mention_type
0,36ecb,entities
104,36ecbplus,entities
217,37ecb,entities
326,37ecbplus,entities
433,38ecb,entities
481,38ecbplus,entities
561,39ecb,entities
703,39ecbplus,entities
813,40ecb,entities
900,40ecbplus,entities


# Score clusters

In [66]:
# from evaluate import load
# coval = load('coval')

In [9]:
# Define the format of the output lines
def format_line(row):
    return f"ECB+/ecbplus_all\t({row['mapped_coref_chain']})"

# Generate the output content
output_lines = ["#begin document (ECB+/ecbplus_all); part 000"]
for _, row in merged_values.iterrows():
    output_lines.append(format_line(row))
output_lines.extend([
    "#end document"
])
# Write the content to a file
with open("key.conll", "w") as f:
    for line in output_lines:
        f.write(line + "\n")

print("File written successfully.")

File written successfully.


In [10]:
list_clusters =["cluster_exp1_seq_sampling",
"cluster_exp1_sim_sampling",
"cluster_exp2_seq_sampling",
"cluster_exp2_sim_sampling",
"cluster_exp3_seq_sampling",
"cluster_exp3_sim_sampling",
"cluster_exp4_seq_sampling",
"cluster_exp4_sim_sampling"    ]
    # Define the format of the output lines
for cluster in list_clusters:
    def format_line(row):
        return f"ECB+/ecbplus_all\t({row[cluster]})"

    # Generate the output content
    output_lines = ["#begin document (ECB+/ecbplus_all); part 000"]
    for _, row in merged_values.iterrows():
        output_lines.append(format_line(row))
    output_lines.extend([
        "#end document"
    ])

    # Write the content to a file
    with open(f"sys_{cluster}.conll", "w") as f:
        for line in output_lines:
            f.write(line + "\n")

    print("File written successfully.")


File written successfully.
File written successfully.
File written successfully.
File written successfully.
File written successfully.
File written successfully.
File written successfully.
File written successfully.
